# Ejercicio 7 — Bloqueo y confusión en un $3^2$ (R)

**Objetivo.** Construir un $3^2$ ejecutado en 3 bloques (lotes) con la relación de
confusión modular $L=x_1+2x_2 \pmod 3$ y medir la ganancia de precisión del bloqueo.

**Factores:** Tiempo de curado ($A$: 10/15/20 min), Temperatura de curado
($B$: 80/100/120 °C)
**Bloque:** lote de adhesivo (3 lotes)
**Respuesta:** Resistencia al corte por solape (MPa)

In [ ]:
df <- read.csv('../../datos/curado-adhesivo-3k-bloques.csv')
cat(sprintf('Corridas: %d (3^2 = 9, repartidas en 3 bloques de 3)\n', nrow(df)))
print(df)

## 1. Verificación del esquema de confusión

In [ ]:
x1_yates <- df$x1 + 1
x2_yates <- df$x2 + 1
L <- (x1_yates + 2*x2_yates) %% 3

verificacion <- data.frame(x1=df$x1, x2=df$x2, L_calculado=L, bloque_dataset=df$bloque)
print(verificacion)
cat(sprintf('\n¿El bloque de cada corrida coincide con L mod 3? %s\n', all(L == df$bloque)))

## 2. Costo de ignorar el bloque

In [ ]:
modelo_sin_bloque <- lm(resistencia ~ x1 + x2 + I(x1^2) + I(x2^2), data = df)
anova_sin <- anova(modelo_sin_bloque)
print(anova_sin)
cat(sprintf('\nMSE (sin bloque): %.4f\n', anova_sin['Residuals', 'Mean Sq']))

## 3. Modelo con bloque como factor de bloqueo

In [ ]:
df$bloque <- factor(df$bloque)
modelo_con_bloque <- lm(resistencia ~ x1 + x2 + I(x1^2) + I(x2^2) + bloque, data = df)
anova_con <- anova(modelo_con_bloque)
print(anova_con)
cat(sprintf('\nMSE (con bloque): %.4f\n', anova_con['Residuals', 'Mean Sq']))

## 4. Comparación de precisión: con y sin bloqueo

In [ ]:
comparacion <- data.frame(
  Modelo = c('Sin bloque', 'Con bloque'),
  MSE_error = c(anova_sin['Residuals', 'Mean Sq'], anova_con['Residuals', 'Mean Sq']),
  gl_error = c(anova_sin['Residuals', 'Df'], anova_con['Residuals', 'Df']),
  p_x1 = c(anova_sin['x1', 'Pr(>F)'], anova_con['x1', 'Pr(>F)']),
  p_x2 = c(anova_sin['x2', 'Pr(>F)'], anova_con['x2', 'Pr(>F)'])
)
print(comparacion)
cat(sprintf('\nEl bloque explica SC=%.3f (p=%.4f): el lote sí introduce una perturbación\n',
            anova_con['bloque', 'Sum Sq'], anova_con['bloque', 'Pr(>F)']))
cat('sistematica real que vale la pena remover del error.\n')

## 5. Qué se sacrifica: la interacción confundida con bloques

El esquema de confusión reparte las 9 corridas usando 2 gl tomados del espacio de la
interacción $AB$ (4 gl en total). Ajustar la interacción **además** del bloque deja el
modelo casi saturado (8 parámetros para 9 corridas, solo 1 gl de error).

In [ ]:
modelo_full <- lm(resistencia ~ x1 + x2 + I(x1^2) + I(x2^2) + x1:x2 + bloque, data = df)
cat(sprintf('Grados de libertad de error con interacción + bloque: %d\n', df.residual(modelo_full)))
print(summary(modelo_full)$coefficients)
cat('\nCon un solo grado de libertad de error, el error estándar de x1:x2 es enorme:\n')
cat('la interacción quedó confundida (parcialmente) con la partición en bloques.\n')

## 6. Conclusión

- El esquema $L=x_1+2x_2 \pmod 3$ reproduce la tabla de la teoría (§7.1) y reparte las 9
  corridas en 3 bloques, dejando $A$ y $B$ (lineal y cuadrático) libres de la variación
  entre lotes.
- Ignorar el bloqueo cuando existe una fuente de variación real infla el error: el MSE
  pasa de $4.04$ a $0.15$ al reconocer el bloque, y los efectos principales pasan de ser
  marginalmente significativos a altamente significativos.
- El precio de esta ganancia es la interacción $AB$: al usarla (parcialmente) para
  definir los bloques, ya no puede estimarse de forma independiente y confiable.
- **Regla práctica.** Bloquea por lote, turno o cualquier factor de perturbación conocido;
  acepta confundir la interacción de mayor orden si —como suele ocurrir— es la menos
  relevante para el objetivo del experimento.